# Chapter 4 dataset generation

**Role:** maintained post-submission guide  
**Execution scope:** documents the matrix and schema; expensive generation calls are interface sketches  
**Thesis section:** 4

The dataset is not a generic preprocessing step: each graph represents one convection–diffusion problem, one mesh, a standard SUPG solution, and a cellwise optimized target. This notebook keeps that mapping explicit. The ignored historical dataset is not included in a fresh clone; use `archive/prototypes/data_generation.ipynb` to audit the submitted process.

## 1. Specify the experimental matrix

The submitted study used **seven benchmark families**, 88 training cases, and 14 test cases. Cases mix triangular and quadrilateral meshes, regular aspect-ratio sweeps, and randomly sized test meshes (30–45 cells per direction). The table below replaces the opaque integer IDs used by the submitted generator.

In [ ]:
benchmark_families = [
    {"legacy_id": 0, "spde": 1, "name": "wedge", "objective": "crosswind residual"},
    {"legacy_id": 1, "spde": 2, "name": "bump", "objective": "crosswind residual"},
    {"legacy_id": 2, "spde": 5, "name": "lifted-edge", "objective": "L2 error (exact solution)"},
    {"legacy_id": 3, "spde": 4, "name": "cylinder", "objective": "L2 error (exact solution)"},
    {"legacy_id": 4, "spde": 3, "name": "falloff", "objective": "crosswind residual"},
    {"legacy_id": 5, "spde": 6, "name": "curved-wave", "objective": "limited residual, t0=0.03"},
    {"legacy_id": 6, "spde": 7, "name": "curved-waves", "objective": "limited residual, t0=1"},
]
benchmark_families


## 2. From FEM fields to graph features

Each DG0 cell becomes a node; shared cell facets create edges. The nine base features are the local PDE coefficients, cell diameter, and standard-SUPG solution/derivatives. Any cell-type indicator used in Chapter 4 must be stored with a name and schema version, rather than appended in an undocumented cell.

In [ ]:
from supgml.data import CaseRepository
from supgml.graph import GraphBuilder
from supgml.graph.features import STANDARD_FEATURES

repository = CaseRepository("data")
builder = GraphBuilder(include_edge_features=False)
print(STANDARD_FEATURES)

# solver = create(case["benchmark"], nx=case["nx"], ny=case["ny"])
# graph = builder.build(solver, target=solver.yh.x.array, problem_id=..., mesh_id=...)
# repository.save(case_id, solver, graph, split=case["split"])


## 3. Generate expensive targets once

Each target is the result of a bounded L-BFGS-B optimization using the discrete-adjoint gradient. This is the computational bottleneck: the thesis reports about 12,000 iterations for SPDE 3 on a 64×8 quadrilateral mesh, while some triangular cases had still not converged after more than one million iterations. Saving the solver state, target, mesh identity, and objective name together is therefore part of reproducibility, not just caching.

## 4. Validate before training

Check the 88/14 split, feature names/order, target and upper-bound shapes, mesh/problem IDs, and graph connectivity. `GraphBuilder` and `CaseRepository` package the stable schema and persistence; this notebook documents why each case exists. The submitted implementation is retained in `archive/prototypes/data_generation.ipynb` for audit, including its original integer IDs and nested adjacency loop.